# Toxic Dataset 

## Loading

In [ ]:
import pandas as pd

# Reading a JSONL file
df = pd.read_json('final_dataset_toxic span/raw_doccano_toxic/all.jsonl', lines=True) #doccano jsonl file input
print(df.head())

      id                                               text Comments  \
0  19666                                main mu kek tai cok       []   
1  19667  user telat ngasih tau elu edan sarap gue berga...       []   
2  19668                      skill cacad ngide jungler cok       []   
3  19669  gw jari lu sebelom gw ukur nyali bacot lu \xf0...       []   
4  19670        maennya goblok banget sih geblek banget dah       []   

                               label  
0                  [[12, 19, TOXIC]]  
1                  [[26, 36, TOXIC]]  
2  [[6, 11, TOXIC], [26, 29, TOXIC]]  
3                  [[33, 38, TOXIC]]  
4  [[8, 14, TOXIC], [26, 32, TOXIC]]  


In [2]:
df.columns

Index(['id', 'text', 'Comments', 'label'], dtype='object')

In [3]:
df = df.drop(columns= "Comments")

In [4]:
df.head()

,id,text,label
0,19666,main mu kek tai cok,"[[12, 19, TOXIC]]"
1,19667,user telat ngasih tau elu edan sarap gue berga...,"[[26, 36, TOXIC]]"
2,19668,skill cacad ngide jungler cok,"[[6, 11, TOXIC], [26, 29, TOXIC]]"
3,19669,gw jari lu sebelom gw ukur nyali bacot lu \xf0...,"[[33, 38, TOXIC]]"
4,19670,maennya goblok banget sih geblek banget dah,"[[8, 14, TOXIC], [26, 32, TOXIC]]"


## convert dataframe into list

In [5]:
text_list = df["text"].tolist()
label_list = df["label"].tolist()

In [6]:
text_list

['main mu kek tai cok',
 'user telat ngasih tau elu edan sarap gue bergaul elu',
 'skill cacad ngide jungler cok',
 'gw jari lu sebelom gw ukur nyali bacot lu \\xf0\\x9f\\x98\\x8f',
 'maennya goblok banget sih geblek banget dah',
 'minum tuh kencing onta emang dasar otak onta',
 'dasar anak babi kau',
 'anak anjing',
 'goblok banget sih mainnya gitu aja kalah mulu',
 'si user kerjaannya delay mulu setan',
 'mainmu noob banget ga bagusnya',
 'parah lo mainnya mati aja',
 'pegawai guardian training kek mentang2 gue kek gembel cuman kaos flat shoe nanya dijutekin bangsad emang male belinya kabur gua matahari',
 'gapernah mendalami alquran ya bang pantesan mulutnya kaya orang kafir bejat hina',
 'anjir kali lihat orang bego debat pilkada kerjanya nyerang karakter shame on you',
 'kau pendikan bodoh',
 'user user si otak tempurung mah ga d ladenin smkn d lawan bloon dungu kalo ga gitu ga makan hehe',
 'budaya kafir bersungguh klau hal2 islam diendahkan',
 'kalahin bangsat pake otak lo napa 

In [7]:
label_list[0]

[[12, 19, 'TOXIC']]

In [8]:
label_list[0][0]

[12, 19, 'TOXIC']

In [9]:
label_list[0][0][0]

12

## Tokenize

In [10]:
# simple tokenization whitespace based and append it into a tokenized list
tokenized_list = []

for text in text_list:
    tokens = text.split()
    tokenized_list.append(tokens)

In [11]:
tokenized_list[1]

['user',
 'telat',
 'ngasih',
 'tau',
 'elu',
 'edan',
 'sarap',
 'gue',
 'bergaul',
 'elu']

In [12]:
label_list[12]

[[47, 53, 'TOXIC'], [91, 98, 'TOXIC']]

## Span and token engineering

In [13]:
# finding the character offset of a token
"""
text : the raw text (ex: "aku suka bunga matahari")
tokens : the tokenized raw text (ex: ["aku", "suka", "bunga", "matahari"])
"""
def get_token_spans(text, tokens):
    spans = []
    current = 0

    for token in tokens:
        # find the first token token to match with current as the start point
        start = text.find(token, current)
        # tag the end of the token's characters with the start character index + the len of the token
        end = start + len(token)
        # mark the character span
        spans.append((start, end))
        # update current into the end
        current = end

    return spans

In [14]:
char_offset_list = []
for text, tokens in zip(text_list, tokenized_list):
    offsets = get_token_spans(text, tokens)
    char_offset_list.append(offsets)


In [15]:
char_offset_list[0]

[(0, 4), (5, 7), (8, 11), (12, 15), (16, 19)]

In [16]:
char_offset_list[0][0]

(0, 4)

In [17]:
char_offset_list[0][0][0]

0

## Convert doccano jsonl into Huggingface BIO scheme

In [18]:
def convert_span_to_bio(tokens, char_offsets, entity):
    tags = ["O"] * len(tokens)
    idx_tags = len(tags)
    for ent in entity:
        start, end, label = ent[0], ent[1], ent[2]
        idx = 0
        for token, offset in zip(tokens, char_offsets):
            off_start, off_end = offset[0], offset[1]
            if off_start == start :
                tags[idx] = f"B-{label}"
            elif off_start > start and off_start < end:
                tags[idx] = f"I-{label}"

    
            idx = idx +1
    return(tags)


In [19]:
convert_span_to_bio(tokenized_list[0],char_offset_list[0], label_list[0] )

['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC']

In [20]:
BIO_label_list = []

for tokens, char_offsets, labels in zip(tokenized_list, char_offset_list, label_list):
    BIO_tags = convert_span_to_bio(tokens, char_offsets, labels)
    BIO_label_list.append(BIO_tags)

In [21]:
BIO_label_list

[['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC'],
 ['O', 'O', 'O', 'O', 'O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'O'],
 ['O', 'B-TOXIC', 'O', 'O', 'B-TOXIC'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC', 'O', 'O'],
 ['O', 'B-TOXIC', 'O', 'O', 'B-TOXIC', 'O', 'O'],
 ['O', 'O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'B-TOXIC', 'I-TOXIC'],
 ['O', 'B-TOXIC', 'I-TOXIC', 'O'],
 ['B-TOXIC', 'I-TOXIC'],
 ['B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'B-TOXIC'],
 ['O', 'B-TOXIC', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-TOXIC',
  'I-TOXIC',
  'I-TOXIC',
  'I-TOXIC'],
 ['B-TOXIC', 'O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'B-TOXIC'],
 ['O',
  'O',
  'O',
  'B-TOXIC',
  'I-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  

In [22]:
import json

def save_to_jsonl(tokens_list, BIO_label_list, path):
    with open(path, "w", encoding="utf-8") as f:
        for tokens, labels in zip(tokens_list, BIO_label_list):
            row = {
                "tokens": tokens,
                "ner_tags": labels
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
save_to_jsonl(tokenized_list, BIO_label_list,"final_dataset_toxic span/Toxic_IOB.jsonl") #Output file

# Non-Toxic Data

## Loading

In [ ]:
import pandas as pd

# Reading a JSONL file
df_neutral = pd.read_csv('splitted dataset/neutral_data_chats.csv') #non doccano file (csv raw file)
print(df_neutral.head())

   Unnamed: 0                                               chat
0           2  kadang berfikir percaya tuhan jatuh berkalikal...
1           6                               gg main lo keren bro
2           7  gue aja kelar rewatch aldnoah zero kampret ema...
3           8  admin belanja port terbaik nak makan ai kepal ...
4          16                            fikiran ampas banget ya


## convert to list

In [25]:
chat_list = df_neutral["chat"].tolist()

In [26]:
# Tokenize
tokenized_chat_list = []
for chat in chat_list:
    tokenized_chat = chat.split()
    tokenized_chat_list.append(tokenized_chat)


In [27]:
BIO_label_list_neutral = []

for tokenized_chat in tokenized_chat_list:
    tags = ["O"] * len(tokenized_chat)
    BIO_label_list_neutral.append(tags)
    


In [28]:
save_to_jsonl(tokenized_chat_list, BIO_label_list_neutral,"final_dataset_toxic span/neutral_iob.jsonl")

# combine 2 jsonl

In [29]:
combined_tokens = tokenized_chat_list + tokenized_list
combined_labels = BIO_label_list_neutral + BIO_label_list

In [30]:
save_to_jsonl(combined_tokens, combined_labels,"final_dataset_toxic span/all_iob.jsonl")

In [31]:
list1 = ['a', 'b']
list2 = ['c', 'd']

labl1 = ['aa', 'bb']
labl2 = ['cc', 'dd']

In [32]:
comlist = list1 + list2
comlabl = labl1 + labl2

In [33]:
comlist

['a', 'b', 'c', 'd']

In [34]:
comlabl

['aa', 'bb', 'cc', 'dd']

# Encoded Labels

In [35]:
combined_labels

[['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O'],
 ['O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O'],
 ['O',
  'O',
  'O',
  'O',


In [36]:
labels = ["O", "B-TOXIC", "I-TOXIC"]
label2id = {"O": 0, "B-TOXIC": 1, "I-TOXIC": 2}
id2label = {0: "O", 1: "B-TOXIC", 2: "I-TOXIC"}

In [37]:
encoded_tags = []
for labels in combined_labels:
    ner_tags_encoded = [label2id[tags] for tags in labels]
    encoded_tags.append(ner_tags_encoded)

In [38]:
combined_labels

[['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O'],
 ['O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O', 'O', 'O', 'O', 'O'],
 ['O', 'O', 'O'],
 ['O',
  'O',
  'O',
  'O',


In [39]:
encoded_tags

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0],
 [0, 0

In [40]:
save_to_jsonl(combined_tokens, encoded_tags,"final_dataset_toxic span/all_iob_encoded.jsonl")

In [1]:
import pandas as pd
print(pd.__version__)

2.2.1
